# C0003R03_PARK25_LC_1D_SCAN_p

SCAN2 泵功率参数扫描程序。

仅保留泵功率/泵能耗计算及其实际使用的输入与中间变量，不进行温度或质量计算。
Excel 输出采用 Times New Roman 字体。


In [19]:
# ============================================================
# Cell 0 - Import Packages and Load BTMS Model
# ============================================================

import os
import sys
import importlib

import numpy as np
import pandas as pd
import CoolProp.CoolProp as CP
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

PROJECT_ROOT = os.path.abspath("../..")

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import lib.BTMS_model as BTMS_model
BTMS_model = importlib.reload(BTMS_model)


In [20]:
# ============================================================
# Cell 1 - SCAN2 Settings
# ============================================================

CASE_ID = 'C0003'
RESULT_XLSX_NAME = 'C0003R03_PARK25_LC_1D_SCAN_p.xlsx'
OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'out', CASE_ID)
os.makedirs(OUTPUT_DIR, exist_ok=True)
RESULT_XLSX_PATH = os.path.join(OUTPUT_DIR, RESULT_XLSX_NAME)

# Fixed SCAN2 operating condition
T_water = 25.0                  # degC
m_dot_total = 0.045             # kg/s, total system mass flow rate
num_parallel_channels = 32

# Coolant / pump parameters used in the power calculation
fluid = 'Water'
p_water = 101325.0              # Pa
pump_efficiency = 0.35
K_minor = 0.0

# Full-mission pump operation time, consistent with the original SCAN2 notebook
pump_operation_time = 1920.0    # s

# SCAN2 geometric scan ranges
W_channel_list = np.array([
    6.0,
    7.0,
    8.0,
    9.0,
    10.0,
], dtype=float) * 1e-3

H_channel_list = np.array([
    2.0,
    3.0,
    4.0,
    5.0,
], dtype=float) * 1e-3

L_channel_list = np.array([
    0.380,
    0.390,
    0.400,
    0.410,
    0.420,
], dtype=float)


In [21]:
# ============================================================
# Cell 2 - Coolant Properties Required by Pump-Power Calculation
# ============================================================

T_water_K = T_water + 273.15

# Only rho and mu are required by the hydraulic/pump-power calculation.
rho_water = CP.PropsSI('D', 'T', T_water_K, 'P', p_water, fluid)  # kg/m3
mu_water = CP.PropsSI('V', 'T', T_water_K, 'P', p_water, fluid)   # Pa s


In [22]:
# ============================================================
# Cell 3 - Pump-Power Calculation for One SCAN2 Geometry
# ============================================================

def calculate_pump_power(W_channel, H_channel, L_channel):
    W_channel = float(W_channel)
    H_channel = float(H_channel)
    L_channel = float(L_channel)

    # 1. Flow split among parallel channels
    m_dot_channel = m_dot_total / num_parallel_channels

    # 2. Rectangular-channel geometry
    A_cool_cs = W_channel * H_channel
    Dh = 2.0 * W_channel * H_channel / (W_channel + H_channel)
    beta = min(W_channel, H_channel) / max(W_channel, H_channel)

    # 3. Channel velocity and Reynolds number
    u_water = m_dot_channel / (rho_water * A_cool_cs)
    Re_water = rho_water * u_water * Dh / mu_water

    # 4. Rectangular-channel Darcy friction factor
    friction_factor = BTMS_model.cal_friction_factor_rect_channel(
        Re_water,
        W_channel,
        H_channel,
    )

    # 5. Pressure drop
    dynamic_pressure = 0.5 * rho_water * u_water**2
    L_over_Dh = L_channel / Dh

    delta_p_major = friction_factor * L_over_Dh * dynamic_pressure
    delta_p_minor = K_minor * dynamic_pressure
    delta_p_total = delta_p_major + delta_p_minor

    # 6. Total volumetric flow rate and pump power
    V_dot_total = m_dot_total / rho_water
    P_pump_W = delta_p_total * V_dot_total / pump_efficiency

    # 7. Pump energy over the full mission
    E_pump_J = P_pump_W * pump_operation_time

    # Save only the inputs/intermediate variables actually used by the
    # pump-power / pump-energy calculation.
    return {
        'W_channel_mm': W_channel * 1e3,
        'H_channel_mm': H_channel * 1e3,
        'L_channel_mm': L_channel * 1e3,
        'T_water_in_C': T_water,
        'm_dot_total_kg_s': m_dot_total,
        'num_parallel_channels': num_parallel_channels,
        'rho_water_kg_m3': rho_water,
        'mu_water_Pa_s': mu_water,
        'm_dot_channel_kg_s': m_dot_channel,
        'A_cool_cs_m2': A_cool_cs,
        'Dh_m': Dh,
        'aspect_ratio_beta': beta,
        'u_water_m_s': u_water,
        'Re_water': Re_water,
        'friction_factor_Darcy': friction_factor,
        'dynamic_pressure_Pa': dynamic_pressure,
        'L_over_Dh': L_over_Dh,
        'K_minor': K_minor,
        'delta_p_major_Pa': delta_p_major,
        'delta_p_minor_Pa': delta_p_minor,
        'delta_p_total_Pa': delta_p_total,
        'V_dot_total_m3_s': V_dot_total,
        'pump_efficiency': pump_efficiency,
        'P_pump_W': P_pump_W,
        'pump_operation_time_s': pump_operation_time,
        'E_pump_J': E_pump_J,
    }


In [23]:
# ============================================================
# Cell 4 - Run SCAN2 Power Scan
# ============================================================

scan2_power_rows = []

total_cases = (
    len(W_channel_list)
    * len(H_channel_list)
    * len(L_channel_list)
)

case_count = 0

for W_channel in W_channel_list:
    for H_channel in H_channel_list:
        for L_channel in L_channel_list:
            case_count += 1

            print(
                f'Running {case_count}/{total_cases}: '
                f'W={W_channel*1e3:.1f} mm, '
                f'H={H_channel*1e3:.1f} mm, '
                f'L={L_channel*1e3:.1f} mm'
            )

            scan2_power_rows.append(
                calculate_pump_power(
                    W_channel,
                    H_channel,
                    L_channel,
                )
            )

print(f'Power scan completed: {len(scan2_power_rows)} cases.')


Running 1/100: W=6.0 mm, H=2.0 mm, L=380.0 mm
Running 2/100: W=6.0 mm, H=2.0 mm, L=390.0 mm
Running 3/100: W=6.0 mm, H=2.0 mm, L=400.0 mm
Running 4/100: W=6.0 mm, H=2.0 mm, L=410.0 mm
Running 5/100: W=6.0 mm, H=2.0 mm, L=420.0 mm
Running 6/100: W=6.0 mm, H=3.0 mm, L=380.0 mm
Running 7/100: W=6.0 mm, H=3.0 mm, L=390.0 mm
Running 8/100: W=6.0 mm, H=3.0 mm, L=400.0 mm
Running 9/100: W=6.0 mm, H=3.0 mm, L=410.0 mm
Running 10/100: W=6.0 mm, H=3.0 mm, L=420.0 mm
Running 11/100: W=6.0 mm, H=4.0 mm, L=380.0 mm
Running 12/100: W=6.0 mm, H=4.0 mm, L=390.0 mm
Running 13/100: W=6.0 mm, H=4.0 mm, L=400.0 mm
Running 14/100: W=6.0 mm, H=4.0 mm, L=410.0 mm
Running 15/100: W=6.0 mm, H=4.0 mm, L=420.0 mm
Running 16/100: W=6.0 mm, H=5.0 mm, L=380.0 mm
Running 17/100: W=6.0 mm, H=5.0 mm, L=390.0 mm
Running 18/100: W=6.0 mm, H=5.0 mm, L=400.0 mm
Running 19/100: W=6.0 mm, H=5.0 mm, L=410.0 mm
Running 20/100: W=6.0 mm, H=5.0 mm, L=420.0 mm
Running 21/100: W=7.0 mm, H=2.0 mm, L=380.0 mm
Running 22/100: W=7.0 

In [24]:
# ============================================================
# Cell 5 - Export Calculation Table
# ============================================================

scan2_power_table = pd.DataFrame(scan2_power_rows)

# Export only the variables actually used in the pump-power / pump-energy
# calculation, including the calculation intermediates.
column_order = [
    'W_channel_mm',
    'H_channel_mm',
    'L_channel_mm',
    'T_water_in_C',
    'm_dot_total_kg_s',
    'num_parallel_channels',
    'rho_water_kg_m3',
    'mu_water_Pa_s',
    'm_dot_channel_kg_s',
    'A_cool_cs_m2',
    'Dh_m',
    'aspect_ratio_beta',
    'u_water_m_s',
    'Re_water',
    'friction_factor_Darcy',
    'dynamic_pressure_Pa',
    'L_over_Dh',
    'K_minor',
    'delta_p_major_Pa',
    'delta_p_minor_Pa',
    'delta_p_total_Pa',
    'V_dot_total_m3_s',
    'pump_efficiency',
    'P_pump_W',
    'pump_operation_time_s',
    'E_pump_J',
]

scan2_power_table = scan2_power_table[column_order]

scan2_power_table.to_excel(
    RESULT_XLSX_PATH,
    sheet_name='Geometry scan_stage2_power',
    index=False,
)

# ============================================================
# Excel formatting: Times New Roman
# ============================================================

wb = load_workbook(RESULT_XLSX_PATH)
ws = wb['Geometry scan_stage2_power']

body_font = Font(name='Times New Roman', size=10)
header_font = Font(name='Times New Roman', size=10, bold=True)
alignment = Alignment(horizontal='center', vertical='center')
thin = Side(style='thin')
border = Border(left=thin, right=thin, top=thin, bottom=thin)

for row in ws.iter_rows():
    for cell in row:
        cell.font = header_font if cell.row == 1 else body_font
        cell.alignment = alignment
        cell.border = border

        if cell.row > 1 and isinstance(cell.value, float):
            cell.number_format = '0.000000'

ws.freeze_panes = 'A2'
ws.auto_filter.ref = ws.dimensions

for col_idx, column_cells in enumerate(ws.columns, start=1):
    max_len = max(
        len(str(cell.value)) if cell.value is not None else 0
        for cell in column_cells
    )
    ws.column_dimensions[get_column_letter(col_idx)].width = min(
        max(max_len + 2, 12),
        28,
    )

wb.save(RESULT_XLSX_PATH)

print(f'Saved: {RESULT_XLSX_PATH}')
scan2_power_table.head()


Saved: d:\eBATS\out\C0003\C0003R03_PARK25_LC_1D_SCAN_p.xlsx


,W_channel_mm,H_channel_mm,L_channel_mm,T_water_in_C,m_dot_total_kg_s,num_parallel_channels,rho_water_kg_m3,mu_water_Pa_s,m_dot_channel_kg_s,A_cool_cs_m2,...,L_over_Dh,K_minor,delta_p_major_Pa,delta_p_minor_Pa,delta_p_total_Pa,V_dot_total_m3_s,pump_efficiency,P_pump_W,pump_operation_time_s,E_pump_J
0,6.0,2.0,380.0,25.0,0.045,32,997.047637,0.00089,0.001406,0.000012,...,126.666667,0.0,151.009791,0.0,151.009791,0.000045,0.35,0.019473,1920.0,37.388229
1,6.0,2.0,390.0,25.0,0.045,32,997.047637,0.00089,0.001406,0.000012,...,130.000000,0.0,154.983733,0.0,154.983733,0.000045,0.35,0.019985,1920.0,38.372130
2,6.0,2.0,400.0,25.0,0.045,32,997.047637,0.00089,0.001406,0.000012,...,133.333333,0.0,158.957675,0.0,158.957675,0.000045,0.35,0.020498,1920.0,39.356031
3,6.0,2.0,410.0,25.0,0.045,32,997.047637,0.00089,0.001406,0.000012,...,136.666667,0.0,162.931617,0.0,162.931617,0.000045,0.35,0.021010,1920.0,40.339932
4,6.0,2.0,420.0,25.0,0.045,32,997.047637,0.00089,0.001406,0.000012,...,140.000000,0.0,166.905559,0.0,166.905559,0.000045,0.35,0.021523,1920.0,41.323832
